In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import optuna
import pacmap
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from pyDRMetrics.pyDRMetrics import DRMetrics

from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 1. Diretórios
# ============================================================

# Se o script estiver dentro da pasta "notebooks" ou "scripts", usamos Path.cwd().parent ou Path.cwd()
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ["notebooks", "scripts"] else Path.cwd()

DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

OPTUNA_DIR = RESULTS_DIR / "optuna"
EMBEDDINGS_DIR = RESULTS_DIR / "embeddings"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

for directory in [OPTUNA_DIR, EMBEDDINGS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

d:\Projetos Python\Mestrado\RBras\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregando os dados

In [2]:
csv_path = DATA_DIR / "df_pca_50_mad5000.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado em: {csv_path}. Execute o script de pré-processamento primeiro.")

print(f"Carregando a base de 50 componentes principais em CSV: {csv_path.name}")
df_pca = pd.read_csv(csv_path, index_col=0)

# Carregamento dos dados clínicos (Subtipos PAM50)
clinical_final = pd.read_csv(
    DATA_DIR / "clinical_final.csv",
    index_col=0
)

# Mantém apenas a coluna de subtipo
clinical_final = clinical_final[["SUBTYPE"]]

# Alinha os dados clínicos mantendo estritamente as amostras aprovadas no QC (presentes em df_pca)
clinical_final = clinical_final.loc[df_pca.index]

# Verificações
assert df_pca.index.equals(clinical_final.index), (
    "Os índices da matriz PCA e da tabela clínica não coincidem."
)

assert not df_pca.isnull().values.any(), (
    "A matriz PCA contém valores ausentes."
)

assert not clinical_final["SUBTYPE"].isnull().any(), (
    "A tabela clínica contém subtipos ausentes."
)

X_data = df_pca.values  # Matriz de 50 PCs que servirá de alta dimensão
y_true = clinical_final["SUBTYPE"].values

print(f"\nNúmero de amostras aprovadas no QC : {X_data.shape[0]}")
print(f"Número de Componentes Principais (PCA) : {X_data.shape[1]}")
print(f"Número de subtipos PAM50           : {clinical_final['SUBTYPE'].nunique()}")

print("\nDistribuição dos subtipos:")
print(clinical_final["SUBTYPE"].value_counts())

Carregando a base de 50 componentes principais em CSV: df_pca_50_mad5000.csv

Número de amostras aprovadas no QC : 964
Número de Componentes Principais (PCA) : 50
Número de subtipos PAM50           : 5

Distribuição dos subtipos:
SUBTYPE
BRCA_LumA      491
BRCA_LumB      192
BRCA_Basal     170
BRCA_Her2       76
BRCA_Normal     35
Name: count, dtype: int64


## Função Objetivo


In [4]:
def executar_pacmap(X, params):
    """
    Executa o PaCMAP utilizando os hiperparâmetros informados.
    """

    reducer = pacmap.PaCMAP(
        n_components=params["n_components"],
        n_neighbors=params["n_neighbors"],
        MN_ratio=params["MN_ratio"],
        FP_ratio=params["FP_ratio"],
        random_state=42
    )

    Z = reducer.fit_transform(X, init="pca")

    return Z


def calcular_metricas_reducao(X, Z):
    """
    Calcula as métricas de preservação da estrutura dos dados.
    """

    dr_metrics = DRMetrics(X, Z)

    q_local = dr_metrics.Qlocal
    auc_qnn = dr_metrics.AUC

    return q_local, auc_qnn


def calcular_metricas_clustering(Z, labels):
    """
    Calcula Silhouette e Calinski-Harabasz.
    """

    n_clusters = len(np.unique(labels))

    if n_clusters < 2 or n_clusters >= len(labels):
        raise ValueError("Número inválido de clusters para avaliação.")

    sil_score = silhouette_score(Z, labels)

    ch_score = calinski_harabasz_score(Z, labels)

    # Mantém a normalização utilizada no código original
    ch_score_norm = ch_score / (ch_score + 1000)

    return sil_score, ch_score, ch_score_norm


def calcular_score_reducao(X, Z):
    """
    Score utilizado para avaliar a redução de dimensionalidade.
    """

    q_local, auc_qnn = calcular_metricas_reducao(X, Z)

    score = 0.5 * q_local + 0.5 * auc_qnn

    return score, q_local, auc_qnn


def calcular_score_clustering(Z, labels):
    """
    Score utilizado para avaliar o clustering.
    """

    sil_score, ch_score, ch_score_norm = (
        calcular_metricas_clustering(Z, labels)
    )

    score = 0.5 * sil_score + 0.5 * ch_score_norm

    return score, sil_score, ch_score, ch_score_norm

In [6]:
def objective_joint(trial):

    # ========================================================
    # Hiperparâmetros do PaCMAP
    # ========================================================

    pacmap_params = {
        "n_neighbors": trial.suggest_int(
            "n_neighbors", 10, 125
        ),

        "MN_ratio": trial.suggest_float(
            "MN_ratio", 0.1, 3.0
        ),

        "FP_ratio": trial.suggest_float(
            "FP_ratio", 0.1, 3.0
        ),

        "n_components": trial.suggest_int(
            "n_components", 2, 10
        )
    }

    # ========================================================
    # Redução de dimensionalidade
    # ========================================================

    Z = executar_pacmap(X_data, pacmap_params)

    try:
        score_dr, q_local, auc_qnn = calcular_score_reducao(
            X_data, Z
        )

    except Exception as error:
        print(f"Erro nas métricas de redução: {error}")
        raise optuna.TrialPruned()

    # ========================================================
    # Hiperparâmetros do GMM
    # ========================================================

    gmm_params = {
        "n_components": trial.suggest_int(
            "gmm_n_components", 3, 7
        ),

        "covariance_type": trial.suggest_categorical(
            "covariance_type",
            ["full", "tied", "diag", "spherical"]
        ),

        "n_init": trial.suggest_int(
            "n_init", 5, 15
        ),

        "random_state": 42
    }

    # ========================================================
    # Clustering
    # ========================================================

    try:
        gmm = GaussianMixture(**gmm_params)

        labels = gmm.fit_predict(Z)

        score_gmm, sil_score, ch_score, ch_score_norm = (
            calcular_score_clustering(Z, labels)
        )

    except Exception as error:
        print(f"Erro no GMM ou nas métricas: {error}")
        raise optuna.TrialPruned()

    # ========================================================
    # Score final
    # ========================================================

    score_final = 0.4 * score_dr + 0.6 * score_gmm

    # Armazena métricas adicionais no ensaio
    trial.set_user_attr("Qlocal", float(q_local))
    trial.set_user_attr("AUC_QNN", float(auc_qnn))
    trial.set_user_attr("Silhouette", float(sil_score))
    trial.set_user_attr("Calinski_Harabasz", float(ch_score))
    trial.set_user_attr("CH_normalizado", float(ch_score_norm))
    trial.set_user_attr("Score_DR", float(score_dr))
    trial.set_user_attr("Score_GMM", float(score_gmm))

    return score_final

## Execução

In [7]:
study_joint = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="Joint_VST_MAD_5000",
    storage=f"sqlite:///{OPTUNA_DIR / 'joint_vst_mad_5000.db'}",
    load_if_exists=True
)

study_joint.optimize(
    objective_joint,
    n_trials=200
)

print("Melhor score:", study_joint.best_value)

print("\nMelhores hiperparâmetros:")
print(study_joint.best_params)

Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly teste

Erro no GMM ou nas métricas: Fitting the mixture model failed because some components have ill-defined empirical covariance (for instance caused by singleton or collapsed samples). Try to decrease the number of components, increase reg_covar, or scale the input data. The numerical accuracy can also be improved by passing float64 data instead of float32.


Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly teste

Melhor score: 0.693578551536384

Melhores hiperparâmetros:
{'n_neighbors': 117, 'MN_ratio': 0.11720476529669058, 'FP_ratio': 2.936948781604621, 'n_components': 7, 'gmm_n_components': 4, 'covariance_type': 'tied', 'n_init': 15}


In [8]:
best_params_joint = study_joint.best_params

# Parâmetros do PaCMAP
best_pacmap_params_joint = {
    "n_neighbors": best_params_joint["n_neighbors"],
    "MN_ratio": best_params_joint["MN_ratio"],
    "FP_ratio": best_params_joint["FP_ratio"],
    "n_components": best_params_joint["n_components"]
}

# Parâmetros do GMM
best_gmm_params_joint = {
    "n_components": best_params_joint["gmm_n_components"],
    "covariance_type": best_params_joint["covariance_type"],
    "n_init": best_params_joint["n_init"],
    "random_state": 42
}

# PaCMAP
Z_joint = executar_pacmap(
    X_data,
    best_pacmap_params_joint
)

# GMM
gmm_joint = GaussianMixture(
    **best_gmm_params_joint
)

labels_joint = gmm_joint.fit_predict(Z_joint)

# Métricas
score_dr_joint, q_local_joint, auc_qnn_joint = (
    calcular_score_reducao(X_data, Z_joint)
)

score_gmm_joint, sil_joint, ch_joint, ch_norm_joint = (
    calcular_score_clustering(Z_joint, labels_joint)
)

ari_joint = adjusted_rand_score(
    y_true,
    labels_joint
)

print("Resultados Joint")
print("------------------------------")
print(f"Qlocal: {q_local_joint:.4f}")
print(f"AUC QNN: {auc_qnn_joint:.4f}")
print(f"Silhouette: {sil_joint:.4f}")
print(f"Calinski-Harabasz: {ch_joint:.4f}")
print(f"ARI: {ari_joint:.4f}")

Note: `n_components != 2` have not been thoroughly tested.


Resultados Joint
------------------------------
Qlocal: 0.4736
AUC QNN: 0.7752
Silhouette: 0.5486
Calinski-Harabasz: 13461.4699
ARI: 0.3960


## Salvando os resultados

In [11]:
# Embedding
df_embedding_joint = pd.DataFrame(
    Z_joint,
    index=df_pca.index,  # <-- Ajustado de df.index para df_pca.index
    columns=[
        f"PaCMAP_{i+1}"
        for i in range(Z_joint.shape[1])
    ]
)

df_embedding_joint["Cluster"] = labels_joint
df_embedding_joint["SUBTYPE"] = y_true

df_embedding_joint.to_csv(
    EMBEDDINGS_DIR / "embedding_joint_vst_mad_5000.csv"
)

# Modelo GMM
joblib.dump(
    gmm_joint,
    EMBEDDINGS_DIR / "gmm_joint_vst_mad_5000.pkl"
)

# Hiperparâmetros
with open(
    OPTUNA_DIR / "best_params_joint_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_params_joint, file, indent=4)

# Histórico dos ensaios
study_joint.trials_dataframe().to_csv(
    OPTUNA_DIR / "trials_joint_vst_mad_5000.csv",
    index=False
)

In [ ]:
best_trial_joint = study_joint.best_trial

# Cria um dicionário consolidado com parâmetros e métricas obtidas
best_info = {
    "number": best_trial_joint.number,
    "value": best_trial_joint.value,
    "params": best_trial_joint.params,
    "user_attrs": best_trial_joint.user_attrs
}

with open(OPTUNA_DIR / "best_trial_info_joint_mad_5000.json", "w") as file:
    json.dump(best_info, file, indent=4)